# 04 — Entraîner et sauvegarder tous les baselines

But : ne pas perdre les modèles entraînés. Chaque baseline est sauvegardé dans `outputs/models/baselines/` avec son encodeur de labels et la liste des features.

In [1]:
import sys
from pathlib import Path
import os
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = c:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured


In [2]:
import pandas as pd
from src.config import FEATURE_DIR, RESULT_DIR
from src.pipeline_steps import step_train_and_save_all_baselines
features = pd.read_parquet(FEATURE_DIR / 'train_audio_features.parquet')
manifest = step_train_and_save_all_baselines(features, preset='fast')
display(manifest)

,model,path
0,dummy_most_frequent,C:\Users\yeyuy\Downloads\projet_ml_audio_class...
1,cosine_knn,C:\Users\yeyuy\Downloads\projet_ml_audio_class...
2,linear_svm_balanced,C:\Users\yeyuy\Downloads\projet_ml_audio_class...
3,sgd_logistic_fast,C:\Users\yeyuy\Downloads\projet_ml_audio_class...
4,extra_trees,C:\Users\yeyuy\Downloads\projet_ml_audio_class...
5,hist_gradient_boosting,C:\Users\yeyuy\Downloads\projet_ml_audio_class...


## Vérifier les scores CV avant de choisir le top 3

In [3]:
cv_results = pd.read_csv(RESULT_DIR / 'model_comparison_cv.csv')
display(cv_results.sort_values('macro_f1_mean', ascending=False))
top3 = cv_results.sort_values('macro_f1_mean', ascending=False).head(3)['model'].tolist()
print('Top 3 candidats:', top3)

,model,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,hist_gradient_boosting,0.734872,0.009089,0.767853,0.004472,0.694628,0.010214
1,extra_trees,0.694365,0.003880,0.724704,0.001953,0.780560,0.006788
2,cosine_knn,0.657985,0.008834,0.653321,0.002743,0.638237,0.010819
3,sgd_logistic_fast,0.341542,0.002257,0.352132,0.000518,0.409466,0.000970
4,linear_svm_balanced,0.340037,0.001759,0.353695,0.001688,0.410144,0.000781
5,dummy_most_frequent,0.000134,0.000000,0.000388,0.000000,0.004854,0.000000


Top 3 candidats: ['hist_gradient_boosting', 'extra_trees', 'cosine_knn']


modèles entraînés et sauvegardés confirment les tendances observées lors de la validation croisée: HistGradientBoosting obtient les meilleurs scores globaux en F1, tandis que ExtraTrees présente la meilleure balanced accuracy, ce qui traduit une meilleure robustesse face au déséquilibre des classes. CosineKNN conserve des performances correctes malgré sa simplicité, indiquant que les descripteurs acoustiques extraits structurent efficacement l’espace des espèces. En revanche, les modèles linéaires restent limités sur ce problème, probablement en raison de relations non linéaires entre les caractéristiques audio et les labels.